# Cisco CX CDC: PySpark Hash-Based CDC (Portable)

**Source**: S3 Parquet (10M rows)  
**CDC**: MD5 hash + full outer join in Spark (reads parquet directly - no pandas conversion)  
**Target**: Snowflake table  
**Runs on**: Snowflake SPCS and EC2/Docker identically

In [ ]:
import importlib, os, subprocess, sys

if not os.path.exists('/usr/lib/jvm'):
    try:
        subprocess.run(['apt-get', 'update', '-qq'], capture_output=True, timeout=60)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'default-jdk'], capture_output=True, timeout=120)
        print('Java installed.')
    except Exception as e:
        print(f'Java install skipped: {e}')

for jvm_path in ['/usr/lib/jvm/java-11-openjdk-amd64', '/usr/lib/jvm/java-17-openjdk-amd64',
                 '/usr/lib/jvm/default-java']:
    if os.path.exists(jvm_path):
        os.environ['JAVA_HOME'] = jvm_path
        break

required = ['pyspark', 'pyarrow']
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing + ['-q'])
    print(f'Installed: {missing}')
else:
    print('All packages available.')
print(f"JAVA_HOME={os.environ.get('JAVA_HOME', 'NOT SET')}")

In [ ]:
import os

def get_snowflake_connection():
    """Connect to Snowflake - works in SPCS or on-prem."""
    try:
        from snowflake.snowpark.context import get_active_session
        session = get_active_session()
        print('Connected via Snowflake SPCS')
        return session, 'SPCS'
    except:
        pass
    import snowflake.connector
    conn = snowflake.connector.connect(connection_name=os.getenv('SNOWFLAKE_CONNECTION_NAME', 'default'))
    print(f'Connected locally to {conn.account}')
    return conn, 'LOCAL'

In [ ]:
import time

S3_BUCKET = 'cisco-cx-cdc-pilot'
COMPARE_COLS = ['software_version', 'cpu_utilization', 'memory_utilization',
                'critical_bugs_count', 'contract_status', 'ip_address']

def detect_runtime():
    try:
        from snowflake.snowpark.context import get_active_session
        get_active_session()
        return 'SPCS'
    except:
        return 'LOCAL'

RUNTIME = detect_runtime()
print(f'Runtime: {RUNTIME}')
print(f'Source: s3://{S3_BUCKET}/cdc_data/')

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import md5, concat_ws, col, lit, when, broadcast

spark = SparkSession.builder \
    .appName('CiscoCX_CDC') \
    .config('spark.driver.memory', '64g') \
    .config('spark.sql.shuffle.partitions', '400') \
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true') \
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'com.amazonaws.auth.EnvironmentVariableCredentialsProvider') \
    .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4') \
    .master('local[*]') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} | Cores: {spark.sparkContext.defaultParallelism}')

t_pipeline_start = time.time()

t0 = time.time()
if RUNTIME == 'SPCS':
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print('Reading from Snowflake External Stage...')
    prev_pdf = session.sql("""
        SELECT $1:device_id::VARCHAR AS device_id, $1:customer_id::VARCHAR AS customer_id,
               $1:software_version::VARCHAR AS software_version, $1:cpu_utilization::FLOAT AS cpu_utilization,
               $1:memory_utilization::FLOAT AS memory_utilization, $1:critical_bugs_count::INTEGER AS critical_bugs_count,
               $1:contract_status::VARCHAR AS contract_status, $1:ip_address::VARCHAR AS ip_address,
               $1:product_family::VARCHAR AS product_family
        FROM @CISCO_CX_PILOT.LANDING_ZONE.CDC_S3_STAGE/prev_snapshot/
    """).to_pandas()
    prev_pdf.columns = [c.lower() for c in prev_pdf.columns]
    curr_pdf = session.sql("""
        SELECT $1:device_id::VARCHAR AS device_id, $1:customer_id::VARCHAR AS customer_id,
               $1:software_version::VARCHAR AS software_version, $1:cpu_utilization::FLOAT AS cpu_utilization,
               $1:memory_utilization::FLOAT AS memory_utilization, $1:critical_bugs_count::INTEGER AS critical_bugs_count,
               $1:contract_status::VARCHAR AS contract_status, $1:ip_address::VARCHAR AS ip_address,
               $1:product_family::VARCHAR AS product_family
        FROM @CISCO_CX_PILOT.LANDING_ZONE.CDC_S3_STAGE/curr_snapshot/
    """).to_pandas()
    curr_pdf.columns = [c.lower() for c in curr_pdf.columns]
    prev_spark = spark.createDataFrame(prev_pdf)
    curr_spark = spark.createDataFrame(curr_pdf)
else:
    print('Reading parquet directly into Spark from S3 (no pandas conversion)...')
    prev_spark = spark.read.parquet(f's3a://{S3_BUCKET}/cdc_data/prev_snapshot/')
    curr_spark = spark.read.parquet(f's3a://{S3_BUCKET}/cdc_data/curr_snapshot/')

t_read = time.time() - t0
prev_count = prev_spark.count()
curr_count = curr_spark.count()
print(f'Read complete: {t_read:.1f}s')
print(f'Previous: {prev_count:,} | Current: {curr_count:,}')

## CDC: Spark Hash Diff + Full Outer Join

MD5 hash of compare columns, full outer join on device_id, classify INSERT/UPDATE/DELETE

In [ ]:
t0 = time.time()
prev_hashed = prev_spark.withColumn('_hash', md5(concat_ws('|', *[col(c).cast('string') for c in COMPARE_COLS])))
curr_hashed = curr_spark.withColumn('_hash', md5(concat_ws('|', *[col(c).cast('string') for c in COMPARE_COLS])))

prev_keys = prev_hashed.select(col('device_id'), col('_hash').alias('_hash_prev'))
curr_keys = curr_hashed.select(col('device_id'), col('_hash').alias('_hash_curr'))
t_hash = time.time() - t0

t0 = time.time()
merged = curr_keys.join(prev_keys, on='device_id', how='full')
merged = merged.withColumn('cdc_op',
    when(col('_hash_prev').isNull(), lit('INSERT'))
    .when(col('_hash_curr').isNull(), lit('DELETE'))
    .when(col('_hash_curr') != col('_hash_prev'), lit('UPDATE'))
    .otherwise(lit('NONE'))
)

changes = merged.filter(col('cdc_op') != 'NONE').cache()
n_inserts = changes.filter(col('cdc_op') == 'INSERT').count()
n_updates = changes.filter(col('cdc_op') == 'UPDATE').count()
n_deletes = changes.filter(col('cdc_op') == 'DELETE').count()
t_join = time.time() - t0

total_cdc = time.time() - t_pipeline_start
print(f'\n=== PySpark Hash-Based CDC Results ===')
print(f'  S3 Read:             {t_read:.1f}s')
print(f'  Hash compute (lazy): {t_hash:.1f}s')
print(f'  Join + Classify:     {t_join:.1f}s')
print(f'  TOTAL CDC:           {total_cdc:.1f}s')
print(f'  Inserts: {n_inserts:,} | Updates: {n_updates:,} | Deletes: {n_deletes:,}')

In [ ]:
t0 = time.time()
conn, runtime = get_snowflake_connection()

upsert_ids = changes.filter(col('cdc_op').isin('INSERT', 'UPDATE')).select('device_id')
upsert_df = curr_hashed.join(broadcast(upsert_ids), on='device_id').drop('_hash').toPandas()
upsert_df.columns = [c.upper() for c in upsert_df.columns]

if runtime == 'LOCAL':
    from snowflake.connector.pandas_tools import write_pandas
    conn.cursor().execute('USE DATABASE CISCO_CX_PILOT')
    conn.cursor().execute('USE SCHEMA PROCESSED')
    conn.cursor().execute('CREATE TABLE IF NOT EXISTS SPARK_CDC_RESULT (DEVICE_ID VARCHAR, CUSTOMER_ID VARCHAR, SOFTWARE_VERSION VARCHAR, CPU_UTILIZATION FLOAT, MEMORY_UTILIZATION FLOAT, CRITICAL_BUGS_COUNT INTEGER, CONTRACT_STATUS VARCHAR, IP_ADDRESS VARCHAR, PRODUCT_FAMILY VARCHAR)')
    conn.cursor().execute('TRUNCATE TABLE SPARK_CDC_RESULT')
    success, nchunks, nrows, _ = write_pandas(conn, upsert_df, 'SPARK_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED')
    print(f'Wrote {nrows:,} rows to Snowflake in {time.time()-t0:.1f}s')
else:
    session = conn
    session.sql('USE DATABASE CISCO_CX_PILOT').collect()
    session.sql('USE SCHEMA PROCESSED').collect()
    session.write_pandas(upsert_df, 'SPARK_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED', overwrite=True)
    print(f'Wrote {len(upsert_df):,} rows to Snowflake in {time.time()-t0:.1f}s')

t_total = time.time() - t_pipeline_start
spark.stop()
print(f'\n=== TOTAL PIPELINE (Read + CDC + Write): {t_total:.1f}s ===')